In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp

spark = SparkSession.builder.getOrCreate()

# Customers
customers = spark.createDataFrame([
    (1, "Asha",   "TX", "2024-01-10"),
    (2, "Rahul",  "CA", "2024-02-15"),
    (3, "Meera",  "NY", "2024-03-05"),
    (4, "John",   "TX", "2024-03-20"),
    (5, "Priya",  "NJ", "2024-04-01"),
], ["customer_id", "customer_name", "state", "signup_date"])

# Orders (some customers have no orders, some orders are cancelled)
orders = spark.createDataFrame([
    (1001, 1, "2024-06-01 10:15:00", "PLACED"),
    (1002, 1, "2024-06-05 12:30:00", "SHIPPED"),
    (1003, 2, "2024-06-02 09:10:00", "CANCELLED"),
    (1004, 3, "2024-06-03 18:20:00", "DELIVERED"),
    (1005, 3, "2024-06-10 20:00:00", "DELIVERED"),
    (1006, 4, "2024-06-07 14:45:00", "SHIPPED"),
], ["order_id", "customer_id", "order_ts", "status"]).withColumn(
    "order_ts", to_timestamp("order_ts")
)

# Order items
order_items = spark.createDataFrame([
    (1001, "P01", 1, 100.0),
    (1001, "P02", 2,  20.0),
    (1002, "P03", 1,  60.0),
    (1003, "P02", 1,  20.0),
    (1004, "P01", 1, 100.0),
    (1004, "P04", 3,  15.0),
    (1005, "P03", 2,  60.0),
    (1006, "P02", 5,  20.0),
], ["order_id", "product_id", "qty", "unit_price"])

# Products
products = spark.createDataFrame([
    ("P01", "Laptop",      "Electronics", 100.0),
    ("P02", "Mouse",       "Electronics",  20.0),
    ("P03", "Headphones",  "Electronics",  60.0),
    ("P04", "Notebook",    "Stationery",   15.0),
], ["product_id", "product_name", "category", "list_price"])

# Register temp views for Spark SQL
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")


In [0]:
%sql
--show all customers from texas
select * from customers
where state = 'TX'

List orders that are not cancelled

In [0]:
%sql
select * from orders
where status != 'CANCELLED'

count orders by status

In [0]:
%sql
select count(*) as num_orders, status from orders
group by status

Compute line total for each order item (qty * unit_price)

In [0]:
%sql
select *, qty * unit_price as line_total from order_items

Total order amount per order_id

In [0]:
%sql
select order_id, sum(qty * unit_price) as order_total from order_items
group by order_id

Customer-wise total spend (exclude CANCELLED orders)

In [0]:
%sql
WITH CTE1 AS (
  SELECT a.customer_id, a.customer_name,b.order_id from customers a join orders b on a.customer_id=b.customer_id
  where b.status != 'CANCELLED'
),
CTE2 AS (
  SELECT order_id, sum(qty * unit_price) as order_total from order_items
group by order_id
)
select a.customer_id, a.customer_name, sum(b.order_total) as customer_total from CTE1 a join cte2 b on a.order_id=b.order_id
group by a.customer_id, a.customer_name
order by customer_total desc

List customers who never placed an order

In [0]:
%sql
select a.customer_id, a.customer_name from customers a left anti join orders b on a.customer_id = b.customer_id